In [ ]:
# GORI SAMPLE LAB — ACE-Step 1.5 | Colab T4
# Notebook corrigido: inicialização com os argumentos básicos oficiais.
# Antes de executar: Ambiente de execução > Alterar tipo > GPU T4.

import os, sys, subprocess, pathlib, shutil

ROOT = "/content/ACE-Step-1.5"

def executar(cmd, cwd=None, env=None):
    print("\n>>>", " ".join(cmd), flush=True)
    p = subprocess.run(cmd, cwd=cwd, env=env, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if p.stdout:
        print(p.stdout[-14000:], flush=True)
    if p.returncode != 0:
        raise RuntimeError("O comando terminou com código " + str(p.returncode) +
                           ". O erro completo está logo acima.")
    return p

print("ETAPA 1/4 — GPU")
gpu = subprocess.run(["bash","-lc","nvidia-smi"], text=True,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(gpu.stdout if gpu.returncode == 0 else "GPU não detectada. Ative T4 GPU nas configurações do Colab.")
if gpu.returncode != 0:
    raise RuntimeError("Ative GPU T4 antes de continuar.")

print("\nETAPA 2/4 — uv e Python 3.12")
executar([sys.executable, "-m", "pip", "install", "-q", "uv"])
executar(["uv", "python", "install", "3.12"])

print("\nETAPA 3/4 — ACE-Step 1.5")
if not pathlib.Path(ROOT).exists():
    executar(["git", "clone", "--depth", "1", "https://github.com/ACE-Step/ACE-Step-1.5.git", ROOT])
executar(["uv", "sync", "--python", "3.12"], cwd=ROOT)

print("\nETAPA 4/4 — Iniciando interface")
print("Na primeira execução, os modelos serão baixados. Aguarde o link gradio.live.", flush=True)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
# Comando básico documentado pelo projeto. Evita flags adicionais que podem variar entre versões.
executar(["uv", "run", "--python", "3.12", "acestep",
          "--server-name", "0.0.0.0", "--port", "7860", "--share"],
         cwd=ROOT, env=env)
